# Simuladores: El problema de la Asignación de Crédito
### Capítulo 8 — *Aprendizaje y Comportamiento Adaptable: Principios y Modelos*
**Arturo Bouzas** · Facultad de Psicología, UNAM · bouzaslab25.com

---
Ejecutar las celdas en orden.


In [ ]:
#@title **Simulador 6.1** — El Gradiente de la Demora
# =============================================================================
# SIMULADORES — CAPÍTULO 6: EL PROBLEMA DE LA ASIGNACIÓN DE CRÉDITO
# Aprendizaje y Comportamiento Adaptable: Principios y Modelos
# Arturo Bouzas
# =============================================================================

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELDA 1 — Simulador 6.1: El Gradiente de la Demora                      ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# ── Paleta claro / oscuro ─────────────────────────────────────────────────────

_PALETAS_61 = {
    'claro': dict(
        azul       = '#2C5282',
        naranja    = '#C05621',
        verde      = '#276749',
        gris       = '#718096',
        fig_bg     = 'white',
        ax_bg      = 'white',
        texto      = '#2D3748',
        legend_bg  = 'white',
        panel_bg   = '#EBF4FF',
        panel_bord = '#2C5282',
        header_bg  = '#2C5282',
        header_fg  = 'white',
        sec_color  = '#2C5282',
    ),
    'oscuro': dict(
        azul       = '#90CDF4',
        naranja    = '#FBD38D',
        verde      = '#9AE6B4',
        gris       = '#A0AEC0',
        fig_bg     = '#1A202C',
        ax_bg      = '#2D3748',
        texto      = '#E2E8F0',
        legend_bg  = '#2D3748',
        panel_bg   = '#2D3748',
        panel_bord = '#4A5568',
        header_bg  = '#1A365D',
        header_fg  = '#EBF8FF',
        sec_color  = '#90CDF4',
    ),
}

def _p_61(oscuro: bool) -> dict:
    return _PALETAS_61['oscuro'] if oscuro else _PALETAS_61['claro']


# ── Modelo ────────────────────────────────────────────────────────────────────

def fuerza_condicionamiento(intervalo, tau):
    """
    Fuerza de condicionamiento como función del intervalo EC–EI.

    Parámetros
    ----------
    intervalo : float  — tiempo entre fin del EC e inicio del EI
                         (negativo = EI antes que EC: sin aprendizaje)
    tau       : float  — constante de tiempo del gradiente

    Retorna
    -------
    float en [0, 1]
    """
    if intervalo < 0:
        return 0.0
    return float(np.exp(-intervalo / tau))


def calcular_gradiente(tau, n_puntos=400):
    """Devuelve (intervalos, fuerzas) para el rango [-0.3τ, 6τ]."""
    intervalos = np.linspace(-0.3 * tau, 6.0 * tau, n_puntos)
    fuerzas    = np.array([fuerza_condicionamiento(t, tau) for t in intervalos])
    return intervalos, fuerzas


# ── HTML de tablas ────────────────────────────────────────────────────────────

def _tabla_html_61(tau_pav, tau_gar, t_medio_pav, t_medio_gar, int_pav, int_gar, v_pav, v_gar, razon, t) -> str:
    """Genera la tabla de resultados con el estilo estandarizado."""
    aviso_pav = ""
    if int_pav < 0:
        aviso_pav = " <br><span style='font-size:10.5px; color:#E53E3E;'>(EI antes que EC → V = 0)</span>"

    aviso_gar = ""
    if int_gar < 0:
        aviso_gar = " <br><span style='font-size:10.5px; color:#E53E3E;'>(EI antes que EC → V = 0)</span>"

    filas = f"""
    <tr style="background:{t['fig_bg']}; color:{t['texto']};">
        <td style="padding:5px 14px; font-weight:bold;">Pavlov</td>
        <td style="text-align:right; padding:5px 14px;">{tau_pav:.0f} s</td>
        <td style="text-align:right; padding:5px 14px;">{t_medio_pav:.1f} s</td>
        <td style="text-align:right; padding:5px 14px;">{int_pav:.0f} s</td>
        <td style="text-align:right; padding:5px 14px; font-weight:bold;">{v_pav:.3f}{aviso_pav}</td>
    </tr>
    <tr style="background:{t['panel_bg']}; color:{t['texto']};">
        <td style="padding:5px 14px; font-weight:bold;">García</td>
        <td style="text-align:right; padding:5px 14px;">{tau_gar:.0f} min</td>
        <td style="text-align:right; padding:5px 14px;">{t_medio_gar:.1f} min</td>
        <td style="text-align:right; padding:5px 14px;">{int_gar:.0f} min</td>
        <td style="text-align:right; padding:5px 14px; font-weight:bold;">{v_gar:.3f}{aviso_gar}</td>
    </tr>
    """

    html = f"""
    <div style="margin-top:20px; font-family:serif;">
      <h4 style="color:{t['azul']}; margin-bottom:6px; font-size:14px;">
        ▸ Resumen del gradiente de la demora
      </h4>
      <p style="color:{t['gris']}; font-size:12px; margin:0 0 10px 0;">
        Comparación directa entre el condicionamiento estándar y la aversión al sabor.
      </p>

      <table style="border-collapse:collapse; font-size:13px; min-width:520px; width:100%;">
        <thead>
          <tr style="background:{t['header_bg']}; color:{t['header_fg']};">
            <th style="padding:8px 14px; text-align:left; border-bottom:2px solid {t['panel_bord']};">Preparación</th>
            <th style="padding:8px 14px; text-align:right; border-bottom:2px solid {t['panel_bord']};">Constante (τ)</th>
            <th style="padding:8px 14px; text-align:right; border-bottom:2px solid {t['panel_bord']};">Vida media (50%)</th>
            <th style="padding:8px 14px; text-align:right; border-bottom:2px solid {t['panel_bord']};">Intervalo de prueba</th>
            <th style="padding:8px 14px; text-align:right; border-bottom:2px solid {t['panel_bord']};">Fuerza final (V)</th>
          </tr>
        </thead>
        <tbody>{filas}</tbody>
      </table>

      <p style="color:{t['gris']}; font-size:11.5px; margin-top:9px;">
        Razón entre las constantes de tiempo: <b>τ_García / τ_Pavlov = {razon:.1f}x</b>
      </p>
    </div>
    """
    return html


# ── Función de graficado ──────────────────────────────────────────────────────

def graficar_61(tau_pav, tau_gar, int_pav, int_gar, tema):
    """Dibuja los dos gradientes (Pavlov y García) con el tema seleccionado."""
    oscuro = (tema == 'Oscuro')
    p      = _p_61(oscuro)

    plt.rcParams.update({
        'font.family'       : 'serif',
        'figure.facecolor'  : p['fig_bg'],
        'axes.facecolor'    : p['ax_bg'],
        'axes.edgecolor'    : p['gris'],
        'axes.spines.top'   : False,
        'axes.spines.right' : False,
        'axes.grid'         : True,
        'grid.alpha'        : 0.30,
        'grid.color'        : p['gris'],
        'axes.labelcolor'   : p['gris'],
        'xtick.color'       : p['gris'],
        'ytick.color'       : p['gris'],
        'text.color'        : p['texto'],
        'legend.facecolor'  : p['legend_bg'],
        'legend.edgecolor'  : p['gris'],
        'legend.labelcolor' : p['texto'],
        'axes.labelsize'    : 11,
        'xtick.labelsize'   : 10,
        'ytick.labelsize'   : 10,
    })

    v_pav = fuerza_condicionamiento(int_pav, tau_pav)
    v_gar = fuerza_condicionamiento(int_gar, tau_gar)
    x1, y1 = calcular_gradiente(tau_pav)
    x2, y2 = calcular_gradiente(tau_gar)
    t_medio_pav = tau_pav * np.log(2)
    t_medio_gar = tau_gar * np.log(2)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7),
                                   gridspec_kw={'wspace': 0.35})
    fig.patch.set_facecolor(p['fig_bg'])
    for ax in (ax1, ax2):
        ax.set_facecolor(p['ax_bg'])

    # ── Panel izquierdo: Pavlov ───────────────────────────────────────────────
    ax1.plot(x1, y1, color=p['azul'], linewidth=2.5, label='Gradiente')
    ax1.axvline(tau_pav, color=p['azul'], linestyle='--', alpha=0.45,
                label=f'tau = {tau_pav:.0f} s')
    ax1.axhline(0.5, color=p['gris'], linestyle='-.', linewidth=1, alpha=0.5,
                label=f'50%  t_medio = {t_medio_pav:.1f} s')

    c1 = p['gris'] if int_pav < 0 else p['azul']
    ax1.axvline(int_pav, color=c1, linestyle=':', linewidth=1.2, alpha=0.7)
    ax1.axhline(v_pav,   color=c1, linestyle=':', linewidth=1.2, alpha=0.7)
    ax1.scatter([int_pav], [v_pav], s=120, zorder=6, color=c1,
                edgecolors='white', linewidths=1.2,
                label=f't = {int_pav:.0f} s  V = {v_pav:.3f}')

    ax1.set_xlabel('Intervalo EC-EI (segundos)', fontsize=11)
    ax1.set_ylabel('Fuerza de condicionamiento', fontsize=11)
    ax1.set_title('Condicionamiento estandar\n(Pavlov: tono -> comida)', fontsize=11)
    ax1.set_ylim(-0.05, 1.15)
    ax1.legend(fontsize=8.5, loc='upper right')

    # ── Panel derecho: García ─────────────────────────────────────────────────
    ax2.plot(x2, y2, color=p['naranja'], linewidth=2.5, label='Gradiente')
    ax2.axvline(tau_gar, color=p['naranja'], linestyle='--', alpha=0.45,
                label=f'tau = {tau_gar:.0f} min')
    ax2.axhline(0.5, color=p['gris'], linestyle='-.', linewidth=1, alpha=0.5,
                label=f'50%  t_medio = {t_medio_gar:.1f} min')

    c2 = p['gris'] if int_gar < 0 else p['naranja']
    ax2.axvline(int_gar, color=c2, linestyle=':', linewidth=1.2, alpha=0.7)
    ax2.axhline(v_gar,   color=c2, linestyle=':', linewidth=1.2, alpha=0.7)
    ax2.scatter([int_gar], [v_gar], s=120, zorder=6, color=c2,
                edgecolors='white', linewidths=1.2,
                label=f't = {int_gar:.0f} min  V = {v_gar:.3f}')

    ax2.set_xlabel('Intervalo EC-EI (minutos)', fontsize=11)
    ax2.set_ylabel('Fuerza de condicionamiento', fontsize=11)
    ax2.set_title('Aversion al sabor\n(Garcia: sabor -> malestar)', fontsize=11)
    ax2.set_ylim(-0.05, 1.15)
    ax2.legend(fontsize=8.5, loc='upper right')

    razon = tau_gar / tau_pav
    fig.suptitle(
        'El gradiente de la demora depende del par estimulo-consecuencia',
        fontsize=12, color=p['gris'], fontweight='bold', y=1.01,
    )
    plt.tight_layout()
    plt.show()

    # ── Visualizar la tabla formateada ────────────────────────────────────────
    html_out = _tabla_html_61(
        tau_pav, tau_gar, t_medio_pav, t_medio_gar,
        int_pav, int_gar, v_pav, v_gar, razon, p
    )
    display(HTML(html_out))


# ── Widgets ───────────────────────────────────────────────────────────────────

_estilo_61     = {'description_width': '140px'}
_estilo_lrg_61 = {'description_width': '160px'}
_layout_61     = widgets.Layout(width='500px')

w_tema_61 = widgets.ToggleButtons(
    options=['Claro', 'Oscuro'],
    value='Claro',
    description='',
    style={'button_width': '120px'},
    layout=widgets.Layout(width='auto'),
)
w_tau_pav_61 = widgets.FloatSlider(
    value=15, min=2, max=120, step=1,
    description='tau Pavlov (s):',
    style=_estilo_61, layout=_layout_61,
    continuous_update=False,
)
w_tau_gar_61 = widgets.FloatSlider(
    value=180, min=30, max=600, step=10,
    description='tau Garcia (min):',
    style=_estilo_61, layout=_layout_61,
    continuous_update=False,
)
w_int_pav_61 = widgets.FloatSlider(
    value=10, min=-10, max=120, step=1,
    description='Intervalo prueba (s):',
    style=_estilo_lrg_61, layout=_layout_61,
    continuous_update=False,
)
w_int_gar_61 = widgets.FloatSlider(
    value=120, min=-60, max=600, step=10,
    description='Intervalo prueba (min):',
    style=_estilo_lrg_61, layout=_layout_61,
    continuous_update=False,
)

btn_reset_61 = widgets.Button(
    description='Restablecer valores',
    button_style='warning',
    layout=widgets.Layout(width='180px'),
)

def _reset_61(_):
    w_tau_pav_61.value = 15
    w_tau_gar_61.value = 180
    w_int_pav_61.value = 10
    w_int_gar_61.value = 120

btn_reset_61.on_click(_reset_61)


# ── HTML de encabezado y secciones ────────────────────────────────────────────

def _html_header_61(oscuro: bool) -> str:
    p = _p_61(oscuro)
    return (
        f'<div style="'
        f'background-color:{p["header_bg"]};'
        f'color:{p["header_fg"]};'
        f'font-family:Georgia,serif;'
        f'font-size:14px;font-weight:bold;'
        f'padding:8px 14px;'
        f'border-radius:6px 6px 0 0;'
        f'letter-spacing:0.5px;">'
        f'&nbsp;Simulador 6.1 &mdash; El Gradiente de la Demora'
        f'</div>'
    )

def _html_sec_61(texto: str, oscuro: bool) -> str:
    p = _p_61(oscuro)
    return (
        f'<div style="'
        f'color:{p["sec_color"]};'
        f'font-family:Georgia,serif;'
        f'font-size:11px;font-weight:bold;'
        f'text-transform:uppercase;letter-spacing:1px;'
        f'margin:8px 0 2px 4px;">{texto}</div>'
    )


# ── Layout de la interfaz ─────────────────────────────────────────────────────

w_header_61 = widgets.HTML(value=_html_header_61(False))
w_sec1_61   = widgets.HTML(value=_html_sec_61('Tema', False))
w_sec2_61   = widgets.HTML(value=_html_sec_61('Preparación Pavlov', False))
w_sec3_61   = widgets.HTML(value=_html_sec_61('Preparación García', False))

_body_layout_61 = widgets.Layout(
    padding='10px 16px 14px 16px',
    background_color=_PALETAS_61['claro']['panel_bg'],
    border=f'1px solid {_PALETAS_61["claro"]["panel_bord"]}',
    border_radius='0 0 6px 6px',
)

_body_61 = widgets.VBox([
    w_sec1_61, w_tema_61,
    w_sec2_61,
    w_tau_pav_61,
    w_int_pav_61,
    w_sec3_61,
    w_tau_gar_61,
    w_int_gar_61,
    widgets.HBox([btn_reset_61]),
], layout=_body_layout_61)

ui_61 = widgets.VBox([w_header_61, _body_61])


def _actualizar_tema_61(change):
    oscuro = (change['new'] == 'Oscuro')
    p      = _p_61(oscuro)
    w_header_61.value = _html_header_61(oscuro)
    w_sec1_61.value   = _html_sec_61('Tema', oscuro)
    w_sec2_61.value   = _html_sec_61('Preparación Pavlov', oscuro)
    w_sec3_61.value   = _html_sec_61('Preparación García', oscuro)
    _body_61.layout.background_color = p['panel_bg']
    _body_61.layout.border           = f'1px solid {p["panel_bord"]}'

w_tema_61.observe(_actualizar_tema_61, names='value')

out_61 = widgets.interactive_output(
    graficar_61,
    {
        'tau_pav' : w_tau_pav_61,
        'tau_gar' : w_tau_gar_61,
        'int_pav' : w_int_pav_61,
        'int_gar' : w_int_gar_61,
        'tema'    : w_tema_61,
    }
)

display(ui_61, out_61)

Output()

In [ ]:
#@title **Simulador 6.2** — Crédito en Ensombrecimiento y Bloqueo
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELDA 2 — Simulador 6.2: Ensombrecimiento y Bloqueo                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

# ── Paleta claro / oscuro ─────────────────────────────────────────────────────

_PALETAS_62 = {
    'claro': dict(
        azul       = '#2C5282',
        naranja    = '#C05621',
        verde      = '#276749',
        gris       = '#718096',
        fig_bg     = 'white',
        ax_bg      = 'white',
        texto      = '#2D3748',
        legend_bg  = '#F7FAFC',
        panel_bg   = '#EBF4FF',
        panel_bord = '#2C5282',
        header_bg  = '#2C5282',
        header_fg  = 'white',
        sec_color  = '#2C5282',
    ),
    'oscuro': dict(
        azul       = '#90CDF4',
        naranja    = '#FBD38D',
        verde      = '#9AE6B4',
        gris       = '#A0AEC0',
        fig_bg     = '#1A202C',
        ax_bg      = '#2D3748',
        texto      = '#E2E8F0',
        legend_bg  = '#2A4365',
        panel_bg   = '#2D3748',
        panel_bord = '#4A5568',
        header_bg  = '#1A365D',
        header_fg  = '#EBF8FF',
        sec_color  = '#90CDF4',
    ),
}

def _p_62(oscuro: bool) -> dict:
    return _PALETAS_62['oscuro'] if oscuro else _PALETAS_62['claro']

# ── Motor interno: Rescorla-Wagner ────────────────────────────────────────────

def _entrenar_62(V_inicial, alpha_A, alpha_B, beta, lam, n_ensayos):
    """
    Entrena un compuesto de dos estímulos A+B registrando el historial ensayo a ensayo.
    """
    V_A, V_B = V_inicial
    hist_A, hist_B = [V_A], [V_B]
    for _ in range(n_ensayos):
        error = lam - (V_A + V_B)
        V_A  += alpha_A * beta * error
        V_B  += alpha_B * beta * error
        hist_A.append(V_A)
        hist_B.append(V_B)
    return np.array(hist_A), np.array(hist_B)

# ── Helpers de diseño ─────────────────────────────────────────────────────────

def _panel_diseno_62(ax, p, titulo, lineas):
    """Dibuja un cuadro de texto lateral explicando las fases y parámetros."""
    ax.axis('off')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    ax.add_patch(mpatches.FancyBboxPatch(
        (0.02, 0.02), 0.96, 0.96,
        boxstyle='round,pad=0.02',
        facecolor=p['ax_bg'], edgecolor=p['gris'], linewidth=1, alpha=0.5
    ))
    ax.text(0.5, 0.96, titulo, ha='center', va='top',
            fontsize=11, fontweight='bold', color=p['texto'],
            transform=ax.transAxes)

    paso = min(0.10, 0.82 / max(len(lineas), 1))
    for i, linea in enumerate(lineas):
        ax.text(0.06, 0.86 - i * paso, linea,
                transform=ax.transAxes, fontsize=10,
                verticalalignment='top', color=p['texto'],
                fontfamily='monospace')


# ── Graficadores por pestaña ──────────────────────────────────────────────────

def _graficar_simultaneo_62(p, salA, salB, beta, lam, n, titulo):
    A, B = _entrenar_62([0.0, 0.0], salA, salB, beta, lam, n)
    ens = np.arange(n + 1)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), gridspec_kw={'width_ratios': [1, 2.2]})
    fig.patch.set_facecolor(p['fig_bg'])
    for ax in axes: ax.set_facecolor(p['ax_bg'])

    _panel_diseno_62(axes[0], p, 'Diseño', [
        '', 'Entrenamiento simultáneo:',
        f'  A + B  →  EI',
        f'  ({n} ensayos)',
        '', 'Parámetros:',
        f'  α_A = {salA:.2f}',
        f'  α_B = {salB:.2f}',
        f'  β   = {beta:.2f}',
        f'  λ   = {lam:.2f}'
    ])

    ax = axes[1]
    ax.plot(ens, A, color=p['azul'], marker='o', ms=4, label='Estímulo A')
    ax.plot(ens, B, color=p['naranja'], marker='o', ms=4, label='Estímulo B')
    ax.axhline(lam, color=p['verde'], ls='--', alpha=0.5, label=f'Asíntota λ = {lam:.1f}')

    if abs(salA - salB) < 0.01 and lam > 0:
        ax.axhline(lam/2, color=p['gris'], ls=':', alpha=0.7, label=f'λ/2 = {lam/2:.2f}')

    ax.set_xlabel('Ensayo')
    ax.set_ylabel('Valor asociativo (V)')
    ax.set_ylim(-0.05, max(1.1, lam * 1.15))
    ax.set_xlim(0, n)

    # Leyenda
    ax.legend(fontsize=9, loc='lower right', framealpha=0.9, edgecolor=p['gris'], facecolor=p['legend_bg'])
    ax.set_title(titulo, fontsize=12, pad=10)

    plt.tight_layout()
    plt.show()

def _graficar_bloqueo_62(p, n1, n2, salA, salB, beta, lam):
    A_f1, B_f1 = _entrenar_62([0.0, 0.0], salA, 0.0, beta, lam, n1)
    A_f2, B_f2 = _entrenar_62([A_f1[-1], 0.0], salA, salB, beta, lam, n2)

    ens = np.arange(n1 + n2 + 1)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), gridspec_kw={'width_ratios': [1, 2.2]})
    fig.patch.set_facecolor(p['fig_bg'])
    for ax in axes: ax.set_facecolor(p['ax_bg'])

    _panel_diseno_62(axes[0], p, 'Diseño Experimental', [
        '', 'Fase 1 (Preentrenamiento):',
        f'  A  →  EI   ({n1} ens.)',
        '', 'Fase 2 (Compuesto):',
        f'  A + B  →  EI ({n2} ens.)',
        '', 'Parámetros:',
        f'  α_A = {salA:.2f}',
        f'  α_B = {salB:.2f}',
        f'  β   = {beta:.2f}',
        f'  λ   = {lam:.2f}'
    ])

    ax = axes[1]
    ax.plot(ens[:n1+1], A_f1, color=p['azul'], marker='o', ms=4, label='Estímulo A (Fase 1)')
    ax.plot(ens[:n1+1], B_f1, color=p['naranja'], alpha=0.3, label='Estímulo B (Ausente)')

    ax.plot(ens[n1:], A_f2, color=p['azul'], marker='s', ms=4, ls='--', alpha=0.8, label='Estímulo A (Fase 2)')
    ax.plot(ens[n1:], B_f2, color=p['naranja'], marker='s', ms=4, label='Estímulo B (Fase 2)')

    ax.axvline(n1, color=p['gris'], ls=':', lw=2, label='Inicio Fase 2')
    ax.axhline(lam, color=p['verde'], ls='--', alpha=0.5, label=f'Asíntota λ = {lam:.1f}')

    ax.set_xlabel('Ensayo')
    ax.set_ylabel('Valor asociativo (V)')
    ax.set_ylim(-0.05, max(1.1, lam * 1.15))
    ax.set_xlim(0, n1 + n2)

    # Leyenda - lower right
    ax.legend(fontsize=8, loc='lower right', framealpha=0.9, edgecolor=p['gris'], facecolor=p['legend_bg'])
    ax.set_title('Bloqueo: A preentrenado bloquea el aprendizaje de B', fontsize=12, pad=10)

    plt.tight_layout()
    plt.show()


# ── Función de despacho ───────────────────────────────────────────────────────

def _dibujar_62(_change=None):
    oscuro = (w_tema_62.value == 'Oscuro')
    p      = _p_62(oscuro)

    plt.rcParams.update({
        'font.family'       : 'serif',
        'figure.facecolor'  : p['fig_bg'],
        'axes.facecolor'    : p['ax_bg'],
        'axes.edgecolor'    : p['gris'],
        'axes.spines.top'   : False,
        'axes.spines.right' : False,
        'axes.grid'         : True,
        'grid.alpha'        : 0.30,
        'grid.color'        : p['gris'],
        'axes.labelcolor'   : p['gris'],
        'xtick.color'       : p['gris'],
        'ytick.color'       : p['gris'],
        'text.color'        : p['texto'],
        'legend.facecolor'  : p['legend_bg'],
        'legend.edgecolor'  : p['gris'],
        'legend.labelcolor' : p['texto'],
        'axes.labelsize'    : 11,
        'xtick.labelsize'   : 10,
        'ytick.labelsize'   : 10,
    })

    with out_62:
        clear_output(wait=True)
        idx = tabs_62.selected_index
        if idx == 0:
            _graficar_simultaneo_62(
                p, sl_ct_salA_62.value, sl_ct_salB_62.value, sl_ct_beta_62.value,
                sl_ct_lam_62.value, sl_ct_n_62.value,
                'Control: Adquisición simultánea equitativa'
            )
        elif idx == 1:
            _graficar_simultaneo_62(
                p, sl_en_salA_62.value, sl_en_salB_62.value, sl_en_beta_62.value,
                sl_en_lam_62.value, sl_en_n_62.value,
                'Ensombrecimiento: El estímulo más saliente captura el valor asociativo'
            )
        elif idx == 2:
            _graficar_bloqueo_62(
                p, sl_bl_n1_62.value, sl_bl_n2_62.value, sl_bl_salA_62.value,
                sl_bl_salB_62.value, sl_bl_beta_62.value, sl_bl_lam_62.value
            )


# ── Widgets ───────────────────────────────────────────────────────────────────

_estilo_62   = {'description_width': '140px'}
_layout_s_62 = widgets.Layout(width='310px')
_layout_l_62 = widgets.Layout(width='400px')

w_tema_62 = widgets.ToggleButtons(
    options=['Claro', 'Oscuro'], value='Claro',
    description='',
    style={'button_width': '120px'},
    layout=widgets.Layout(width='auto'),
)

sl_ct_salA_62 = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description='Saliencia A (α_A):', style=_estilo_62, layout=_layout_l_62, continuous_update=False)
sl_ct_salB_62 = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description='Saliencia B (α_B):', style=_estilo_62, layout=_layout_l_62, continuous_update=False)
sl_ct_n_62    = widgets.IntSlider(value=30, min=1, max=100, step=1, description='Ensayos:', style=_estilo_62, layout=_layout_s_62, continuous_update=False)
sl_ct_lam_62  = widgets.FloatSlider(value=1.0, min=0.1, max=2.0, step=0.1, description='Asíntota (λ):', style=_estilo_62, layout=_layout_s_62, continuous_update=False)
sl_ct_beta_62 = widgets.FloatSlider(value=0.5, min=0.1, max=1.0, step=0.05, description='Tasa EI (β):', style=_estilo_62, layout=_layout_s_62, continuous_update=False)

sl_en_salA_62 = widgets.FloatSlider(value=0.9, min=0.0, max=1.0, step=0.05, description='Saliencia A (α_A):', style=_estilo_62, layout=_layout_l_62, continuous_update=False)
sl_en_salB_62 = widgets.FloatSlider(value=0.1, min=0.0, max=1.0, step=0.05, description='Saliencia B (α_B):', style=_estilo_62, layout=_layout_l_62, continuous_update=False)
sl_en_n_62    = widgets.IntSlider(value=30, min=1, max=100, step=1, description='Ensayos:', style=_estilo_62, layout=_layout_s_62, continuous_update=False)
sl_en_lam_62  = widgets.FloatSlider(value=1.0, min=0.1, max=2.0, step=0.1, description='Asíntota (λ):', style=_estilo_62, layout=_layout_s_62, continuous_update=False)
sl_en_beta_62 = widgets.FloatSlider(value=0.5, min=0.1, max=1.0, step=0.05, description='Tasa EI (β):', style=_estilo_62, layout=_layout_s_62, continuous_update=False)

sl_bl_salA_62 = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description='Saliencia A (α_A):', style=_estilo_62, layout=_layout_l_62, continuous_update=False)
sl_bl_salB_62 = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description='Saliencia B (α_B):', style=_estilo_62, layout=_layout_l_62, continuous_update=False)
sl_bl_n1_62   = widgets.IntSlider(value=30, min=1, max=100, step=1, description='Ensayos Fase 1:', style=_estilo_62, layout=_layout_s_62, continuous_update=False)
sl_bl_n2_62   = widgets.IntSlider(value=30, min=1, max=100, step=1, description='Ensayos Fase 2:', style=_estilo_62, layout=_layout_s_62, continuous_update=False)
sl_bl_lam_62  = widgets.FloatSlider(value=1.0, min=0.1, max=2.0, step=0.1, description='Asíntota (λ):', style=_estilo_62, layout=_layout_s_62, continuous_update=False)
sl_bl_beta_62 = widgets.FloatSlider(value=0.5, min=0.1, max=1.0, step=0.05, description='Tasa EI (β):', style=_estilo_62, layout=_layout_s_62, continuous_update=False)

btn_reset_62 = widgets.Button(
    description='Restablecer valores',
    button_style='warning',
    layout=widgets.Layout(width='200px', height='32px', margin='15px 0 0 0'),
)

def _reset_62(_):
    sl_ct_salA_62.value = 0.5; sl_ct_salB_62.value = 0.5; sl_ct_n_62.value = 30; sl_ct_lam_62.value = 1.0; sl_ct_beta_62.value = 0.5
    sl_en_salA_62.value = 0.9; sl_en_salB_62.value = 0.1; sl_en_n_62.value = 30; sl_en_lam_62.value = 1.0; sl_en_beta_62.value = 0.5
    sl_bl_salA_62.value = 0.5; sl_bl_salB_62.value = 0.5; sl_bl_n1_62.value = 30; sl_bl_n2_62.value = 30; sl_bl_lam_62.value = 1.0; sl_bl_beta_62.value = 0.5

btn_reset_62.on_click(_reset_62)


# ── Tabs ──────────────────────────────────────────────────────────────────────

tab_ct_62 = widgets.VBox([
    widgets.HTML('<p style="margin:0 0 10px 0;"><b>Control:</b> Ambos estímulos se presentan juntos desde el inicio sin historia previa. Tienen la misma saliencia y compiten de forma simétrica.</p>'),
    widgets.HBox([sl_ct_salA_62, sl_ct_salB_62]),
    widgets.HBox([sl_ct_n_62, sl_ct_lam_62, sl_ct_beta_62]),
])

tab_en_62 = widgets.VBox([
    widgets.HTML('<p style="margin:0 0 10px 0;"><b>Ensombrecimiento:</b> Un estímulo es más saliente que el otro (α_A > α_B). Observa cómo el estímulo saliente acapara el crédito asociativo.</p>'),
    widgets.HBox([sl_en_salA_62, sl_en_salB_62]),
    widgets.HBox([sl_en_n_62, sl_en_lam_62, sl_en_beta_62]),
])

tab_bl_62 = widgets.VBox([
    widgets.HTML('<p style="margin:0 0 10px 0;"><b>Bloqueo:</b> A se pre-entrena solo (Fase 1) hasta predecir el EI. Luego se entrena el compuesto A+B (Fase 2). Como A ya predice el EI, el error de predicción es nulo y B no aprende.</p>'),
    widgets.HBox([sl_bl_salA_62, sl_bl_salB_62]),
    widgets.HBox([sl_bl_n1_62, sl_bl_n2_62]),
    widgets.HBox([sl_bl_lam_62, sl_bl_beta_62, widgets.HTML('<div style="width:10px"></div>')]),
])

tabs_62 = widgets.Tab(children=[tab_ct_62, tab_en_62, tab_bl_62])
tabs_62.set_title(0, 'Control')
tabs_62.set_title(1, 'Ensombrecimiento')
tabs_62.set_title(2, 'Bloqueo')


# ── (sliders + pestaña + tema) ─────────────────────────

sliders_62 = [
    sl_ct_salA_62, sl_ct_salB_62, sl_ct_n_62, sl_ct_lam_62, sl_ct_beta_62,
    sl_en_salA_62, sl_en_salB_62, sl_en_n_62, sl_en_lam_62, sl_en_beta_62,
    sl_bl_salA_62, sl_bl_salB_62, sl_bl_n1_62, sl_bl_n2_62, sl_bl_lam_62, sl_bl_beta_62
]

for _sl in sliders_62:
    _sl.observe(_dibujar_62, names='value')

tabs_62.observe(_dibujar_62, names='selected_index')
w_tema_62.observe(_dibujar_62, names='value')


# ── HTML de encabezado y secciones ────────────────────────────────────────────

def _html_header_62(oscuro: bool) -> str:
    p = _p_62(oscuro)
    return (
        f'<div style="'
        f'background-color:{p["header_bg"]};'
        f'color:{p["header_fg"]};'
        f'font-family:Georgia,serif;'
        f'font-size:14px;font-weight:bold;'
        f'padding:8px 14px;'
        f'border-radius:6px 6px 0 0;'
        f'letter-spacing:0.5px;">'
        f'&nbsp;Simulador 6.2 &mdash; Ensombrecimiento y Bloqueo (Curvas)'
        f'</div>'
    )

def _html_sec_62(texto: str, oscuro: bool) -> str:
    p = _p_62(oscuro)
    return (
        f'<div style="'
        f'color:{p["sec_color"]};'
        f'font-family:Georgia,serif;'
        f'font-size:11px;font-weight:bold;'
        f'text-transform:uppercase;letter-spacing:1px;'
        f'margin:8px 0 2px 4px;">{texto}</div>'
    )


# ── Layout de la interfaz ─────────────────────────────────────────────────────

w_header_62 = widgets.HTML(value=_html_header_62(False))
w_sec1_62   = widgets.HTML(value=_html_sec_62('Tema', False))
w_sec2_62   = widgets.HTML(value=_html_sec_62('Parámetros y Fases', False))

_body_layout_62 = widgets.Layout(
    padding='10px 16px 14px 16px',
    background_color=_PALETAS_62['claro']['panel_bg'],
    border=f'1px solid {_PALETAS_62["claro"]["panel_bord"]}',
    border_radius='0 0 6px 6px',
)

_body_62 = widgets.VBox([
    w_sec1_62, w_tema_62,
    w_sec2_62, tabs_62,
    btn_reset_62
], layout=_body_layout_62)

ui_62 = widgets.VBox([w_header_62, _body_62])


def _actualizar_panel_62(change):
    oscuro = (change['new'] == 'Oscuro')
    p      = _p_62(oscuro)
    w_header_62.value = _html_header_62(oscuro)
    w_sec1_62.value   = _html_sec_62('Tema', oscuro)
    w_sec2_62.value   = _html_sec_62('Parámetros y Fases', oscuro)
    _body_62.layout.background_color = p['panel_bg']
    _body_62.layout.border           = f'1px solid {p["panel_bord"]}'

w_tema_62.observe(_actualizar_panel_62, names='value')

out_62 = widgets.Output()

display(ui_62, out_62)
_dibujar_62()

Output()

In [ ]:
#@title **Simulador 6.3** — Ensombrecimiento y Bloqueo: Resultados y Curvas
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELDA 3 — Simulador 6.3: Ensombrecimiento y Bloqueo (Integrado)         ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

# ── Paleta claro / oscuro ─────────────────────────────────────────────────────

_PALETAS_63 = {
    'claro': dict(
        azul       = '#2C5282',
        naranja    = '#C05621',
        verde      = '#276749',
        gris       = '#718096',
        fig_bg     = 'white',
        ax_bg      = 'white',
        texto      = '#2D3748',
        legend_bg  = '#F7FAFC',
        panel_bg   = '#EBF4FF',
        panel_bord = '#2C5282',
        header_bg  = '#2C5282',
        header_fg  = 'white',
        sec_color  = '#2C5282',
        res_bg     = '#F8FAFC',
    ),
    'oscuro': dict(
        azul       = '#90CDF4',
        naranja    = '#FBD38D',
        verde      = '#9AE6B4',
        gris       = '#A0AEC0',
        fig_bg     = '#1A202C',
        ax_bg      = '#2D3748',
        texto      = '#E2E8F0',
        legend_bg  = '#2A4365',
        panel_bg   = '#2D3748',
        panel_bord = '#4A5568',
        header_bg  = '#1A365D',
        header_fg  = '#EBF8FF',
        sec_color  = '#90CDF4',
        res_bg     = '#2D3748',
    ),
}

def _p_63(oscuro: bool) -> dict:
    return _PALETAS_63['oscuro'] if oscuro else _PALETAS_63['claro']

# Colores auxiliares
_COLOR_A_LITE_63 = '#A8C4EF'
_COLOR_B_LITE_63 = '#EFB0B0'
_UMBRAL_CERO_63  = 0.02
_BETA_63         = 0.4


# ── Motor interno: Rescorla-Wagner (Actualizado con historial) ────────────────

def _aprender_solo_63(V0, alpha, beta, lam, n):
    """Entrena un único EC durante n ensayos y guarda el historial."""
    V = float(V0)
    hist = [V]
    for _ in range(n):
        V += alpha * beta * max(0.0, lam - V)
        hist.append(V)
    return V, np.array(hist)


def _aprender_compuesto_63(V_A0, V_B0, alpha_A, alpha_B, beta, lam, n):
    """Entrena el compuesto A+B durante n ensayos y guarda el historial."""
    V_A, V_B = float(V_A0), float(V_B0)
    hist_A, hist_B = [V_A], [V_B]
    for _ in range(n):
        error = max(0.0, lam - (V_A + V_B))
        V_A  += alpha_A * beta * error
        V_B  += alpha_B * beta * error
        hist_A.append(V_A)
        hist_B.append(V_B)
    return V_A, V_B, np.array(hist_A), np.array(hist_B)


# ── Helpers de barras, diseño y texto de resultados ───────────────────────────

def _barra_63(ax, x, altura, color, etiqueta, width=0.35, alpha=0.85):
    ax.bar(x, altura, color=color, alpha=alpha, width=width,
           edgecolor='white', linewidth=1.4, label=etiqueta)
    offset = max(altura * 0.03, 0.01)
    ax.text(x, altura + offset, f'{altura:.2f}',
            ha='center', va='bottom', fontsize=11,
            fontweight='bold', color=color)


def _barra_o_cero_63(ax, x, altura, color, etiqueta, ylim_max, width=0.35):
    if altura < _UMBRAL_CERO_63:
        ax.bar(x, ylim_max * 0.012, color=color, alpha=0.5, width=width,
               edgecolor='white', linewidth=1.0, label=etiqueta)
        ax.annotate(
            '≈ 0\n(bloqueo\ncompleto)',
            xy=(x, ylim_max * 0.015),
            xytext=(x + 0.18, ylim_max * 0.18),
            fontsize=8.5, color=color, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=color, lw=1.2),
            ha='left', va='bottom',
        )
    else:
        _barra_63(ax, x, altura, color, etiqueta, width=width)


def _formatear_eje_63(ax, titulo, ylim, p, ylabel='Respuesta condicionada (prueba)'):
    """Aplica el formato estándar a los ejes de barras, incluyendo la leyenda abajo."""
    ax.set_ylim(*ylim)
    ax.set_title(titulo, fontsize=11, pad=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_xticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.legend(fontsize=9, loc='upper center', bbox_to_anchor=(0.5, -0.16),
              framealpha=0.9, facecolor=p['legend_bg'], edgecolor=p['gris'])


def _panel_diseno_63(ax, titulo, lineas, p, colores=None):
    ax.axis('off')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    if colores is None:
        colores = [p['texto']] * len(lineas)
    ax.add_patch(mpatches.FancyBboxPatch(
        (0.02, 0.02), 0.96, 0.96,
        boxstyle='round,pad=0.02',
        facecolor=p['res_bg'], edgecolor=p['panel_bord'], linewidth=1, alpha=0.5
    ))
    ax.text(0.5, 0.96, titulo, ha='center', va='top',
            fontsize=11, fontweight='bold', color=p['texto'],
            transform=ax.transAxes)
    paso = min(0.10, 0.80 / max(len(lineas), 1))
    for i, (linea, color) in enumerate(zip(lineas, colores)):
        ax.text(0.06, 0.86 - i * paso, linea,
                transform=ax.transAxes, fontsize=9,
                verticalalignment='top', color=color,
                fontfamily='monospace')


def _mostrar_resultados_63(p, titulo, lineas):
    """Genera un cuadro HTML elegante para mostrar las conclusiones teóricas."""
    html_content = "".join([f"<div style='margin-bottom: 6px;'>{l}</div>" for l in lineas])
    html = f"""
    <div style="background-color:{p['res_bg']}; border-left: 4px solid {p['azul']};
                padding: 12px 16px; margin-top: 15px; border-radius: 4px;
                box-shadow: 0 1px 3px rgba(0,0,0,0.1); border: 1px solid {p['panel_bord']};">
        <div style="color:{p['azul']}; font-family:Georgia,serif; font-size:13px; font-weight:bold;
                    text-transform:uppercase; letter-spacing:0.5px; margin-bottom:8px;">
            {titulo}
        </div>
        <div style="color:{p['texto']}; font-family:sans-serif; font-size:13px; line-height:1.5;">
            {html_content}
        </div>
    </div>
    """
    display(widgets.HTML(value=html))

# ── Graficadores por pestaña ──────────────────────────────────────────────────

def _graficar_linea_base_63(p, sal, n, sbi):
    V, hist_V = _aprender_solo_63(0.0, sal, _BETA_63, sbi, n)
    ylim = (0, sbi * 1.35)

    fig, axes = plt.subplots(1, 3, figsize=(14, 5.5),
                             gridspec_kw={'width_ratios': [0.8, 1.5, 1.0]})
    fig.patch.set_facecolor(p['fig_bg'])
    for ax in axes: ax.set_facecolor(p['ax_bg'])

    _panel_diseno_63(axes[0], 'Diseño experimental', [
        '', 'Entrenamiento:',
        '  EC  →  SBI',
        f'  ({n} ensayos)',
        f'  saliencia = {sal:.2f}',
        '', 'Prueba:',
        '  EC solo  →  ¿Respuesta?',
    ], p, colores=[p['texto']]*8)

    ax_c = axes[1]
    ens = np.arange(n + 1)
    ax_c.plot(ens, hist_V, color=p['azul'], marker='o', ms=4, label='EC')
    ax_c.axhline(sbi, color=p['verde'], ls='--', alpha=0.5, label='λ (SBI)')
    ax_c.set_title('Curva de Aprendizaje', fontsize=11, pad=10)
    ax_c.set_xlabel('Ensayo', fontsize=10)
    ax_c.set_ylabel('Valor (V)', fontsize=10)
    ax_c.set_ylim(-0.05, max(1.1, sbi * 1.15))
    ax_c.legend(fontsize=9, loc='upper center', bbox_to_anchor=(0.5, -0.16),
                ncol=2, framealpha=0.9, facecolor=p['legend_bg'], edgecolor=p['gris'])

    axes[2].set_xlim(0, 1)
    _barra_63(axes[2], 0.5, V, p['azul'], f'EC  (V = {V:.2f})')
    axes[2].axhline(sbi, color=p['gris'], linestyle='--', alpha=0.5,
                    label=f'Valor máximo = {sbi:.1f}')
    _formatear_eje_63(axes[2], 'Resultado final en prueba', ylim, p)

    fig.suptitle('Línea Base: un solo EC',
                 fontsize=13, fontweight='bold', color=p['texto'], y=1.01)

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.28)
    plt.show()

    _mostrar_resultados_63(p, "Interpretación de Resultados", [
        f"• <b>Valor asociativo final (V):</b> {V:.3f} (Máximo posible λ = {sbi:.1f})",
        f"• <b>Porcentaje de aprendizaje completado:</b> {(V / sbi * 100) if sbi > 0 else 0:.1f}%"
    ])


def _graficar_ensombrecimiento_63(p, salA, salB, n, sbi):
    V_A_ctrl, _ = _aprender_solo_63(0.0, salA, _BETA_63, sbi, n)
    V_B_ctrl, _ = _aprender_solo_63(0.0, salB, _BETA_63, sbi, n)
    V_A_exp, V_B_exp, hist_A_exp, hist_B_exp = _aprender_compuesto_63(
        0.0, 0.0, salA, salB, _BETA_63, sbi, n)

    ylim = (0, sbi * 1.40)
    fig, axes = plt.subplots(1, 4, figsize=(16, 5.5),
                             gridspec_kw={'width_ratios': [0.8, 1.5, 1.0, 1.0]})
    fig.patch.set_facecolor(p['fig_bg'])
    for ax in axes: ax.set_facecolor(p['ax_bg'])

    _panel_diseno_63(axes[0], 'Diseño', [
        '', 'Ctrl A:   A solo → SBI',
        f'          ({n} ens., sal={salA:.2f})',
        '', 'Ctrl B:   B solo → SBI',
        f'          ({n} ens., sal={salB:.2f})',
        '', 'Exp:      A + B  → SBI',
        f'          ({n} ens.)',
        '', 'Prueba:   A solo / B solo',
    ], p, colores=[p['texto']]*10)

    ax_c = axes[1]
    ens = np.arange(n + 1)
    ax_c.plot(ens, hist_A_exp, color=p['azul'], marker='o', ms=4, label='Estímulo A')
    ax_c.plot(ens, hist_B_exp, color=p['naranja'], marker='o', ms=4, label='Estímulo B')
    ax_c.axhline(sbi, color=p['verde'], ls='--', alpha=0.5, label='λ (SBI)')
    ax_c.set_title('Curvas de Aprendizaje\n(Grupo Exp. A+B)', fontsize=11, pad=10)
    ax_c.set_xlabel('Ensayo', fontsize=10)
    ax_c.set_ylabel('Valor (V)', fontsize=10)
    ax_c.set_ylim(-0.05, max(1.1, sbi * 1.15))
    ax_c.legend(fontsize=9, loc='upper center', bbox_to_anchor=(0.5, -0.16),
                ncol=2, framealpha=0.9, facecolor=p['legend_bg'], edgecolor=p['gris'])

    for ax, (vA, vB), titulo in [
        (axes[2], (V_A_ctrl, V_B_ctrl), 'Grupos Control\n(EC solos)'),
        (axes[3], (V_A_exp,  V_B_exp),  'Grupo Experimental\n(A+B juntos)'),
    ]:
        _barra_63(ax, 0.3, vA, p['azul'],    f'A  V={vA:.2f}')
        _barra_63(ax, 0.7, vB, p['naranja'], f'B  V={vB:.2f}')
        ax.axhline(sbi, color=p['gris'], ls='--', alpha=0.4, label=f'máx = {sbi:.1f}')
        ax.set_xlim(0, 1)
        _formatear_eje_63(ax, titulo, ylim, p)

    fig.suptitle('Ensombrecimiento: el estímulo más saliente captura el crédito',
                 fontsize=13, fontweight='bold', color=p['texto'], y=1.01)

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.28)
    plt.show()

    red_A = (V_A_ctrl - V_A_exp) / V_A_ctrl * 100 if V_A_ctrl > 0 else 0
    red_B = (V_B_ctrl - V_B_exp) / V_B_ctrl * 100 if V_B_ctrl > 0 else 0

    lineas_res = [
        f"• <b>Aprendizaje en A:</b> Control = {V_A_ctrl:.3f} | Compuesto = {V_A_exp:.3f} <i>(Reducción: {red_A:.1f}%)</i>",
        f"• <b>Aprendizaje en B:</b> Control = {V_B_ctrl:.3f} | Compuesto = {V_B_exp:.3f} <i>(Reducción: {red_B:.1f}%)</i>",
    ]

    if red_B > red_A + 5:
        lineas_res.append(f"<span style='color:{p['naranja']}; font-weight:bold;'>→ Conclusión: B queda ensombrecido por A (la mayor saliencia de A acapara el crédito).</span>")
    elif red_A > red_B + 5:
        lineas_res.append(f"<span style='color:{p['azul']}; font-weight:bold;'>→ Conclusión: A queda ensombrecido por B (la mayor saliencia de B acapara el crédito).</span>")
    else:
        lineas_res.append(f"<span style='color:{p['verde']}; font-weight:bold;'>→ Conclusión: Competencia equilibrada (las saliencias similares reparten el crédito).</span>")

    _mostrar_resultados_63(p, "Análisis de Ensombrecimiento", lineas_res)


def _graficar_bloqueo_63(p, n1, n2, sal, sbi1, sbi2):
    V_A_ctrl, V_B_ctrl, _, _ = _aprender_compuesto_63(
        0.0, 0.0, sal, sal, _BETA_63, sbi2, n2)
    V_A_f1, hist_A_f1 = _aprender_solo_63(0.0, sal, _BETA_63, sbi1, n1)
    V_A_end, V_B_end, hist_A_f2, hist_B_f2 = _aprender_compuesto_63(
        V_A_f1, 0.0, sal, sal, _BETA_63, sbi2, n2)

    delta_A_f2 = V_A_end - V_A_f1
    delta_B_f2 = V_B_end
    ylim       = (0, max(sbi1, sbi2) * 1.40)

    colores_diseno = (
        [p['texto']] * 2 + [p['gris']] * 4 +
        [p['texto']] + [p['naranja']] * 5 + [p['texto']]
    )

    fig, axes = plt.subplots(1, 4, figsize=(16, 5.5),
                             gridspec_kw={'width_ratios': [0.8, 1.5, 0.9, 1.1]})
    fig.patch.set_facecolor(p['fig_bg'])
    for ax in axes: ax.set_facecolor(p['ax_bg'])

    _panel_diseno_63(axes[0], 'Diseño experimental', [
        '', 'CONTROL:',
        '  Fase 1: —',
        f'  Fase 2: A+B → SBI ({n2} ens.)',
        '  Prueba: B solo',
        '', 'EXPERIMENTAL:',
        f'  Fase 1: A → SBI ({n1} ens.)',
        f'  Fase 2: A+B → SBI ({n2} ens.)',
        '  Prueba: B solo',
        '', f'  Saliencia A = B = {sal:.2f}',
    ], p, colores=colores_diseno)

    # ── PANEL 1: CURVAS ──
    ax_c = axes[1]
    ax_c.plot(np.arange(n1+1), hist_A_f1, color=p['azul'], marker='o', ms=4, label='A (Fase 1)')
    ax_c.plot(np.arange(n1+1), np.zeros(n1+1), color=p['naranja'], alpha=0.3, label='B (Ausente)')
    ax_c.plot(np.arange(n1, n1+n2+1), hist_A_f2, color=p['azul'], marker='s', ms=4, ls='--', alpha=0.8, label='A (Fase 2)')
    ax_c.plot(np.arange(n1, n1+n2+1), hist_B_f2, color=p['naranja'], marker='s', ms=4, label='B (Fase 2)')
    ax_c.axvline(n1, color=p['gris'], ls=':', lw=2, label='Inicio F2')
    ax_c.axhline(sbi2, color=p['verde'], ls='--', alpha=0.5, label=f'λ F2 = {sbi2:.1f}')
    if abs(sbi2 - sbi1) > 0.05:
        ax_c.axhline(sbi1, color=p['verde'], ls=':', alpha=0.4, label=f'λ F1 = {sbi1:.1f}')

    ax_c.set_title('Curvas de Aprendizaje\n(Grupo Experimental)', fontsize=11, pad=10)
    ax_c.set_xlabel('Ensayo', fontsize=10)
    ax_c.set_ylabel('Valor (V)', fontsize=10)
    ax_c.set_xlim(0, n1 + n2)
    ax_c.set_ylim(-0.05, ylim[1] * 0.85)

    ax_c.legend(fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.16),
                ncol=2, framealpha=0.9, facecolor=p['legend_bg'], edgecolor=p['gris'])

    # ── PANEL 2: RESULTADO EN PRUEBA B ──
    ax2 = axes[2]
    _barra_63(ax2, 0.3, V_B_ctrl, p['gris'],
              f'Control  V_B={V_B_ctrl:.2f}')
    _barra_o_cero_63(ax2, 0.7, V_B_end, p['naranja'],
                     f'Experimental  V_B={V_B_end:.2f}', ylim[1])
    ax2.axhline(sbi2, color=p['gris'], ls='--', alpha=0.4,
                label=f'máx = {sbi2:.1f}')
    ax2.set_xlim(0, 1)
    _formatear_eje_63(ax2, 'Resultado en prueba\n(solo estímulo B)', ylim, p)

    # ── PANEL 3: REPARTO DE CRÉDITO ──
    ax3 = axes[3]
    ax3.bar(0.5, V_A_f1, color=p['azul'], alpha=0.85, width=0.45,
            edgecolor='white', linewidth=1.4,
            label=f'A tras Fase 1 = {V_A_f1:.2f}')
    ax3.bar(0.5, delta_A_f2, bottom=V_A_f1,
            color=_COLOR_A_LITE_63, alpha=0.85, width=0.45,
            edgecolor='white', linewidth=1.4,
            label=f'A ganó en Fase 2 = {delta_A_f2:.2f}')
    ax3.bar(0.5, delta_B_f2, bottom=V_A_f1 + delta_A_f2,
            color=_COLOR_B_LITE_63, alpha=0.85, width=0.45,
            edgecolor='white', linewidth=1.4,
            label=f'B ganó en Fase 2 = {delta_B_f2:.2f}')

    tope = V_A_f1 + delta_A_f2 + delta_B_f2
    ax3.text(0.5, tope + tope * 0.03 + 0.01,
             f'ΣV = {tope:.2f}', ha='center', fontsize=9,
             color='#555555', style='italic')
    ax3.axhline(sbi2, color=p['gris'], ls='--', alpha=0.5,
                label=f'SBI Fase 2 = {sbi2:.1f}')
    if abs(sbi2 - sbi1) > 0.05:
        ax3.axhline(sbi1, color=p['gris'], ls=':', alpha=0.6,
                    label=f'SBI Fase 1 = {sbi1:.1f}')

    ax3.set_xlim(0, 1)
    ax3.set_ylim(*ylim)
    ax3.set_xticks([])
    ax3.set_ylabel('Valor acumulado', fontsize=10)
    ax3.set_title('Reparto del crédito en Fase 2\n(lo que aprendió cada EC)', fontsize=11, pad=10)
    ax3.spines['top'].set_visible(False)
    ax3.spines['right'].set_visible(False)
    ax3.grid(axis='y', alpha=0.3, linestyle='--')

    ax3.legend(fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.16),
               ncol=1, framealpha=0.9, facecolor=p['legend_bg'], edgecolor=p['gris'])

    fig.suptitle('Bloqueo: si A ya predice el SBI, B no aprende nada nuevo',
                 fontsize=13, fontweight='bold', color=p['texto'], y=1.01)

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.28)
    plt.show()

    bloqueo_pct     = (1 - V_B_end / V_B_ctrl) * 100 if V_B_ctrl > 0 else 100
    error_inicio_f2 = max(0.0, sbi2 - V_A_f1)

    lineas_res = [
        f"• <b>V_A al final de la Fase 1:</b> {V_A_f1:.3f} (Máximo λ₁ = {sbi1:.1f})",
        f"• <b>Error de predicción disponible al inicio de Fase 2 (λ₂ − V_A):</b> {error_inicio_f2:.3f}",
        f"• <b>Valor ganado en Fase 2:</b> A = {delta_A_f2:.3f} | B = {delta_B_f2:.3f}",
        f"• <b>V_B control (expectativa de aprendizaje sin bloqueo):</b> {V_B_ctrl:.3f}",
        f"• <b>Efecto del Bloqueo:</b> B aprendió solo el <b>{100 - bloqueo_pct:.1f}%</b> de lo que aprendería si estuviera solo."
    ]

    if error_inicio_f2 < 0.05:
        lineas_res.append(f"<span style='color:{p['naranja']}; font-weight:bold;'>→ Conclusión: Bloqueo Completo. A ya predecía todo el SBI. El error de predicción es nulo (≈ 0), por lo que nadie aprende nada nuevo.</span>")
    elif sbi2 > sbi1 + 0.05:
        lineas_res.append(f"<span style='color:{p['verde']}; font-weight:bold;'>→ Conclusión: Desbloqueo. El aumento de intensidad en F2 (SBI₂ > SBI₁) genera un error de predicción positivo. Esto 'abre' crédito asociativo que B puede capturar.</span>")
    else:
        lineas_res.append(f"<span style='color:{p['azul']}; font-weight:bold;'>→ Conclusión: Bloqueo Parcial. A no había saturado λ en la Fase 1, dejando una fracción de error de predicción disponible para que B lo adquiera.</span>")

    _mostrar_resultados_63(p, "Análisis de Bloqueo", lineas_res)


# ── Función de despacho ───────────────────────────────────────────────────────

def _dibujar_63(_change=None):
    oscuro = (w_tema_63.value == 'Oscuro')
    p      = _p_63(oscuro)

    plt.rcParams.update({
        'font.family'       : 'serif',
        'figure.facecolor'  : p['fig_bg'],
        'axes.facecolor'    : p['ax_bg'],
        'axes.edgecolor'    : p['gris'],
        'axes.spines.top'   : False,
        'axes.spines.right' : False,
        'axes.grid'         : True,
        'grid.alpha'        : 0.30,
        'grid.color'        : p['gris'],
        'axes.labelcolor'   : p['gris'],
        'xtick.color'       : p['gris'],
        'ytick.color'       : p['gris'],
        'text.color'        : p['texto'],
        'legend.facecolor'  : p['legend_bg'],
        'legend.edgecolor'  : p['gris'],
        'legend.labelcolor' : p['texto'],
        'axes.labelsize'    : 11,
        'xtick.labelsize'   : 10,
        'ytick.labelsize'   : 10,
    })

    with out_63:
        clear_output(wait=True)
        idx = tabs_63.selected_index
        if idx == 0:
            _graficar_linea_base_63(
                p, sl_lb_sal_63.value, sl_lb_n_63.value, sl_lb_sbi_63.value,
            )
        elif idx == 1:
            _graficar_ensombrecimiento_63(
                p, sl_en_salA_63.value, sl_en_salB_63.value, sl_en_n_63.value, sl_en_sbi_63.value,
            )
        elif idx == 2:
            _graficar_bloqueo_63(
                p, sl_bl_n1_63.value, sl_bl_n2_63.value, sl_bl_sal_63.value, sl_bl_sbi1_63.value, sl_bl_sbi2_63.value,
            )


# ── Widgets ───────────────────────────────────────────────────────────────────

_estilo_63   = {'description_width': '160px'}
_layout_s_63 = widgets.Layout(width='420px')
_layout_l_63 = widgets.Layout(width='460px')

w_tema_63 = widgets.ToggleButtons(
    options=['Claro', 'Oscuro'], value='Claro',
    description='', style={'button_width': '120px'}, layout=widgets.Layout(width='auto')
)

sl_lb_sal_63 = widgets.FloatSlider(value=0.6, min=0.1, max=1.0, step=0.05, description='Saliencia EC:', style=_estilo_63, layout=_layout_l_63, continuous_update=False)
sl_lb_n_63   = widgets.IntSlider(value=20, min=3, max=80, step=1, description='Ensayos:', style=_estilo_63, layout=_layout_s_63, continuous_update=False)
sl_lb_sbi_63 = widgets.FloatSlider(value=1.0, min=0.2, max=2.0, step=0.1, description='Intensidad SBI:', style=_estilo_63, layout=_layout_s_63, continuous_update=False)

sl_en_salA_63 = widgets.FloatSlider(value=0.8, min=0.1, max=1.0, step=0.05, description='Saliencia A:', style=_estilo_63, layout=_layout_l_63, continuous_update=False)
sl_en_salB_63 = widgets.FloatSlider(value=0.2, min=0.1, max=1.0, step=0.05, description='Saliencia B:', style=_estilo_63, layout=_layout_l_63, continuous_update=False)
sl_en_n_63    = widgets.IntSlider(value=20, min=3, max=80, step=1, description='Ensayos:', style=_estilo_63, layout=_layout_s_63, continuous_update=False)
sl_en_sbi_63  = widgets.FloatSlider(value=1.0, min=0.2, max=2.0, step=0.1, description='Intensidad SBI:', style=_estilo_63, layout=_layout_s_63, continuous_update=False)

sl_bl_n1_63   = widgets.IntSlider(value=25, min=0, max=80, step=1, description='Ensayos fase 1:', style=_estilo_63, layout=_layout_s_63, continuous_update=False)
sl_bl_n2_63   = widgets.IntSlider(value=20, min=3, max=80, step=1, description='Ensayos fase 2:', style=_estilo_63, layout=_layout_s_63, continuous_update=False)
sl_bl_sal_63  = widgets.FloatSlider(value=0.6, min=0.1, max=1.0, step=0.05, description='Saliencia A y B:', style=_estilo_63, layout=_layout_l_63, continuous_update=False)
sl_bl_sbi1_63 = widgets.FloatSlider(value=1.0, min=0.2, max=2.0, step=0.1, description='Intensidad SBI f1:', style=_estilo_63, layout=_layout_s_63, continuous_update=False)
sl_bl_sbi2_63 = widgets.FloatSlider(value=1.0, min=0.2, max=2.0, step=0.1, description='Intensidad SBI f2:', style=_estilo_63, layout=_layout_s_63, continuous_update=False)

btn_reset_63 = widgets.Button(description='Restablecer valores', button_style='warning', layout=widgets.Layout(width='200px', height='32px'))

def _reset_63(_):
    sl_lb_sal_63.value  = 0.6;  sl_lb_n_63.value    = 20;  sl_lb_sbi_63.value  = 1.0
    sl_en_salA_63.value = 0.8;  sl_en_salB_63.value = 0.2; sl_en_n_63.value    = 20; sl_en_sbi_63.value = 1.0
    sl_bl_n1_63.value   = 25;   sl_bl_n2_63.value   = 20;  sl_bl_sal_63.value  = 0.6; sl_bl_sbi1_63.value = 1.0; sl_bl_sbi2_63.value = 1.0

btn_reset_63.on_click(_reset_63)


# ── Tabs ──────────────────────────────────────────────────────────────────────

tab_lb_63 = widgets.VBox([
    widgets.HTML('<b>Línea base:</b> un solo estímulo contiguo al SBI. Referencia para comparar ensombrecimiento y bloqueo.'),
    sl_lb_sal_63, sl_lb_n_63, sl_lb_sbi_63,
])

tab_en_63 = widgets.VBox([
    widgets.HTML('<b>Ensombrecimiento:</b> dos estímulos A y B presentados juntos. Varía la saliencia relativa para ver cuánto captura cada uno.'),
    sl_en_salA_63, sl_en_salB_63, sl_en_n_63, sl_en_sbi_63,
])

tab_bl_63 = widgets.VBox([
    widgets.HTML('<b>Bloqueo:</b> A se preentrenan solo (Fase 1); luego A+B juntos (Fase 2). Modifica la intensidad del SBI en Fase 2 para explorar el efecto de <i>bloqueo</i>.'),
    sl_bl_n1_63, sl_bl_n2_63, sl_bl_sal_63, sl_bl_sbi1_63, sl_bl_sbi2_63,
])

tabs_63 = widgets.Tab(children=[tab_lb_63, tab_en_63, tab_bl_63])
tabs_63.set_title(0, 'Línea Base')
tabs_63.set_title(1, 'Ensombrecimiento')
tabs_63.set_title(2, 'Bloqueo')


# ── (sliders + pestaña + tema) ─────────────────────────

for _sl in [sl_lb_sal_63, sl_lb_n_63, sl_lb_sbi_63,
            sl_en_salA_63, sl_en_salB_63, sl_en_n_63, sl_en_sbi_63,
            sl_bl_n1_63, sl_bl_n2_63, sl_bl_sal_63, sl_bl_sbi1_63, sl_bl_sbi2_63]:
    _sl.observe(_dibujar_63, names='value')

tabs_63.observe(_dibujar_63, names='selected_index')
w_tema_63.observe(_dibujar_63, names='value')


# ── HTML de encabezado y secciones ────────────────────────────────────────────

def _html_header_63(oscuro: bool) -> str:
    p = _p_63(oscuro)
    return (f'<div style="background-color:{p["header_bg"]}; color:{p["header_fg"]}; font-family:Georgia,serif; font-size:14px; font-weight:bold; padding:8px 14px; border-radius:6px 6px 0 0; letter-spacing:0.5px;">&nbsp;Simulador 6.3 &mdash; Ensombrecimiento y Bloqueo Integrado</div>')

def _html_sec_63(texto: str, oscuro: bool) -> str:
    p = _p_63(oscuro)
    return (f'<div style="color:{p["sec_color"]}; font-family:Georgia,serif; font-size:11px; font-weight:bold; text-transform:uppercase; letter-spacing:1px; margin:8px 0 2px 4px;">{texto}</div>')


# ── Layout de la interfaz ─────────────────────────────────────────────────────

w_header_63 = widgets.HTML(value=_html_header_63(False))
w_sec1_63   = widgets.HTML(value=_html_sec_63('Tema', False))
w_sec2_63   = widgets.HTML(value=_html_sec_63('Parámetros', False))

_body_layout_63 = widgets.Layout(
    padding='10px 16px 14px 16px', background_color=_PALETAS_63['claro']['panel_bg'],
    border=f'1px solid {_PALETAS_63["claro"]["panel_bord"]}', border_radius='0 0 6px 6px',
)

_body_63 = widgets.VBox([
    w_sec1_63, w_tema_63,
    w_sec2_63, tabs_63,
    widgets.HBox([btn_reset_63]),
], layout=_body_layout_63)

ui_63 = widgets.VBox([w_header_63, _body_63])


def _actualizar_panel_63(change):
    oscuro = (change['new'] == 'Oscuro')
    p      = _p_63(oscuro)
    w_header_63.value = _html_header_63(oscuro)
    w_sec1_63.value   = _html_sec_63('Tema', oscuro)
    w_sec2_63.value   = _html_sec_63('Parámetros', oscuro)
    _body_63.layout.background_color = p['panel_bg']
    _body_63.layout.border           = f'1px solid {p["panel_bord"]}'

w_tema_63.observe(_actualizar_panel_63, names='value')

out_63 = widgets.Output()

display(ui_63, out_63)
_dibujar_63()

Output()

---
## Créditos y licencia

Este notebook es parte del proyecto:

> **Bouzas, A. (2026).** *Aprendizaje y Comportamiento Adaptable: Principios y Modelos.*
> Lab25, Facultad de Psicología, UNAM.
> https://www.bouzaslab25.com

Apoyo en la construcción del simulador: **Eduardo Sánchez**.

Código disponible en: **https://github.com/bouzaslab25/libro-aca**
Licencia: [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)
